# Chapter 4: Creating and Using a MSA Dataset

## Introduction

Multiple Sequence Alignments (MSAs) are essential tools in bioinformatics for comparing and analyzing related protein or nucleic acid sequences. The `MSADataset` class in the PG2 dataset system provides functionality for working with MSA data in various formats. This chapter explains how to create, load, and use MSA datasets in your projects.

## Understanding the MSADataset

The `MSADataset` class is designed to handle multiple sequence alignment data from various file formats. It provides functionality for:

1. Loading MSA data from different file formats (A2M, A3M, PSI)
2. Parsing and organizing sequence records
3. Accessing aligned sequences
4. Extracting record names from different header formats

## Supported MSA Formats

The `MSADataset` supports three common MSA file formats:

1. **A2M**: Aligned FASTA format where gaps in the query sequence are represented by dashes (-) and gaps in the target sequences are represented by lowercase letters
2. **A3M**: Similar to A2M but with lowercase letters in the target sequences removed
3. **PSI**: A simple format where each line contains a header and a sequence separated by whitespace

## Creating an MSA Dataset

There are two main ways to create an MSA dataset:

### 1. From a dataset.toml File

The most common approach is to create an MSA dataset through a `dataset.toml` file:

In [ ]:
import os

# First, let's create a simple A3M file for demonstration
a3m_content = '''\
>seq1 description of sequence 1
MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG
>seq2 description of sequence 2
MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAaG
>seq3 description of sequence 3
MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGa
'''

# Create directory if it doesn't exist
os.makedirs("../example_data", exist_ok=True)

# Write the A3M file
a3m_path = "../example_data/sample_alignment.a3m"
with open(a3m_path, "w") as f:
    f.write(a3m_content)

print(f"Created sample A3M file at {os.path.abspath(a3m_path)}")

# Now create a dataset.toml file that references this MSA
dataset_toml_content = '''
[resources]
records = "../example_data/sample_data.csv"  # Path to a CSV file
msa = "../example_data/sample_alignment.a3m"

[records]
columns = ["sequence", "score"]
sequence_feature = "sequence"

[metadata]
name = "Sample MSA Dataset"
description = "A sample dataset with MSA for demonstration purposes"
'''

# Write the content to a file
with open("sample_msa_dataset.toml", "w") as f:
    f.write(dataset_toml_content)

print(f"Created sample_msa_dataset.toml at {os.path.abspath('sample_msa_dataset.toml')}")

# Create a simple CSV file for the records
import pandas as pd

data = {
    'sequence': ['MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG'],
    'score': [1.0]
}

df = pd.DataFrame(data)
csv_path = "../example_data/sample_data.csv"
df.to_csv(csv_path, index=False)

# Now try to load the dataset from the TOML file
from pg2_dataset.dataset import Dataset

try:
    # Load dataset from TOML file
    dataset = Dataset.from_toml("sample_msa_dataset.toml")
    
    # Access the MSA dataset
    # Note: As of the current implementation, MSA might not be automatically loaded
    # from the dataset.toml file, so we'll show the direct method below
    print("\nLoaded dataset from TOML file")
    print(f"Dataset name: {dataset.meta.name}")
except Exception as e:
    print(f"\nError loading dataset: {e}")

### 2. Directly Creating an MSADataset

You can also create an `MSADataset` directly:

In [ ]:
try:
    from pg2_dataset.backends import MSADataset
    
    # Create the dataset with a file path
    msa_dataset = MSADataset(file_path="../example_data/sample_alignment.a3m")
    
    print("Successfully created MSA dataset directly")
    print(f"Number of sequences: {len(msa_dataset.sequences)}")
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Error creating dataset: {e}")

## Accessing MSA Data

Once you've loaded an MSA dataset, you can access the data in several ways:

### Accessing All Sequences

In [ ]:
try:
    if 'msa_dataset' in locals() and hasattr(msa_dataset, 'sequences'):
        # Get all sequences in the MSA
        sequences = msa_dataset.sequences
        print(f"Number of sequences in alignment: {len(sequences)}")
        
        # Print the first few sequences
        for i, seq in enumerate(sequences[:3]):
            print(f"Sequence {i+1}: {seq[:50]}..." if len(seq) > 50 else f"Sequence {i+1}: {seq}")
            
        # Print sequence lengths
        print("\nSequence lengths:")
        for i, seq in enumerate(sequences[:3]):
            print(f"Sequence {i+1}: {len(seq)} residues")
    else:
        print("MSA dataset not available or no sequences loaded")
except Exception as e:
    print(f"Error accessing sequences: {e}")

### Accessing the MSA Dictionary

In [ ]:
try:
    if 'msa_dataset' in locals() and hasattr(msa_dataset, 'msa'):
        # Access the raw MSA dictionary (record_name -> sequence)
        msa_dict = msa_dataset.msa
        
        print("MSA records:")
        # Iterate through all records
        for record_name, sequence in msa_dict.items():
            print(f"Record: {record_name}, Sequence length: {len(sequence)}")
            print(f"Sequence start: {sequence[:30]}...")
            print()
    else:
        print("MSA dataset not available or no MSA dictionary loaded")
except Exception as e:
    print(f"Error accessing MSA dictionary: {e}")

## Understanding MSA File Parsing

The `MSADataset` class handles the parsing of MSA files through specialized methods for each format:

### A2M Format

A2M files are parsed line by line, with sequences being built up from non-header lines:

```
>seq1 description
ABCDEFGHI
>seq2 description
ABC--FGHI
```

The parser ensures that all sequences in an A2M file have the same length, which is a requirement for this format.

In [ ]:
# Let's create a simple A2M file
a2m_content = '''\
>seq1 description of sequence 1
MKTVRQERLKSIVRILERSKEPVSGAQ
>seq2 description of sequence 2
MKTVRQERLKSIVRILERSKEPVSGAQ
>seq3 description of sequence 3
MKTVRQERLKSIVRILERSKEPVSGAQ
'''

a2m_path = "../example_data/sample_alignment.a2m"
with open(a2m_path, "w") as f:
    f.write(a2m_content)

print(f"Created sample A2M file at {os.path.abspath(a2m_path)}")

try:
    # Load the A2M file
    a2m_dataset = MSADataset(file_path=a2m_path)
    
    print(f"\nLoaded A2M dataset with {len(a2m_dataset.sequences)} sequences")
    print(f"All sequences have the same length: {len(set(len(seq) for seq in a2m_dataset.sequences)) == 1}")
except Exception as e:
    print(f"Error loading A2M file: {e}")

### A3M Format

A3M files are parsed similarly to A2M files, but without the length validation:

```
>seq1 description
ABCDEFGHI
>seq2 description
ABCdeFGHI  # lowercase letters represent insertions relative to the query
```

In [ ]:
# We already created an A3M file earlier, let's load it again
try:
    a3m_dataset = MSADataset(file_path="../example_data/sample_alignment.a3m")
    
    print(f"Loaded A3M dataset with {len(a3m_dataset.sequences)} sequences")
    
    # Show the sequences
    for i, (name, seq) in enumerate(a3m_dataset.msa.items()):
        print(f"Sequence {i+1} ({name}): {seq}")
except Exception as e:
    print(f"Error loading A3M file: {e}")

### PSI Format

PSI files have a simpler format where each line contains a header and sequence:

```
seq1 ABCDEFGHI
seq2 ABC--FGHI
```

In [ ]:
# Let's create a simple PSI file
psi_content = '''\
seq1 MKTVRQERLKSIVRILERSKEPVSGAQ
seq2 MKTVRQERLKSIVRILERSKEPVSGAQ
seq3 MKTVRQERLKSIVRILERSKEPVSGAQ
'''

psi_path = "../example_data/sample_alignment.psi"
with open(psi_path, "w") as f:
    f.write(psi_content)

print(f"Created sample PSI file at {os.path.abspath(psi_path)}")

try:
    # Load the PSI file
    psi_dataset = MSADataset(file_path=psi_path)
    
    print(f"\nLoaded PSI dataset with {len(psi_dataset.sequences)} sequences")
    
    # Show the sequences
    for i, (name, seq) in enumerate(psi_dataset.msa.items()):
        print(f"Sequence {i+1} ({name}): {seq}")
except Exception as e:
    print(f"Error loading PSI file: {e}")

## Record Name Extraction

The `MSADataset` includes a helper method for extracting record names from FASTA headers:

In [ ]:
try:
    # Test the record name extraction method
    if 'MSADataset' in locals():
        # For a standard FASTA header
        record_name = MSADataset._extract_record_name(">sequence1 description")
        print(f"Standard header: '{record_name}'")
        
        # For a UniProt-style header
        record_name = MSADataset._extract_record_name(">tr|A0A1B2C3D4|PROTEIN_NAME description")
        print(f"UniProt header: '{record_name}'")
    else:
        print("MSADataset class not available")
except Exception as e:
    print(f"Error testing record name extraction: {e}")

## Example Usage

Here's a complete example of working with an MSA dataset:

In [ ]:
try:
    from pg2_dataset.backends import MSADataset
    
    # Load MSA from file
    msa_dataset = MSADataset(file_path="../example_data/sample_alignment.a3m")
    
    # Check basic information
    print(f"Number of sequences: {len(msa_dataset.sequences)}")
    
    # Get the first few sequences
    first_sequences = msa_dataset.sequences[:3]
    for i, seq in enumerate(first_sequences):
        print(f"Sequence {i+1}: {seq[:50]}..." if len(seq) > 50 else f"Sequence {i+1}: {seq}")
    
    # Find the most common residue at each position
    if msa_dataset.sequences:
        seq_length = len(msa_dataset.sequences[0])
        for pos in range(min(10, seq_length)):  # First 10 positions
            residues = [seq[pos] for seq in msa_dataset.sequences if len(seq) > pos]
            residue_counts = {}
            for res in residues:
                residue_counts[res] = residue_counts.get(res, 0) + 1
            
            most_common = max(residue_counts.items(), key=lambda x: x[1])
            print(f"Position {pos+1}: Most common residue is {most_common[0]} ({most_common[1]} occurrences)")
except Exception as e:
    print(f"Error in example usage: {e}")

## Working with Large MSAs

MSA files can be very large, containing thousands of sequences. Here are some tips for working with large MSAs:

In [ ]:
# This is a demonstration of how you would work with large MSAs
# We'll use our small example file but show the pattern

try:
    # Load the MSA
    msa_dataset = MSADataset(file_path="../example_data/sample_alignment.a3m")
    
    # Get basic statistics
    sequence_count = len(msa_dataset.sequences)
    print(f"Total sequences: {sequence_count}")
    
    # Calculate sequence length statistics
    lengths = [len(seq) for seq in msa_dataset.sequences]
    avg_length = sum(lengths) / len(lengths) if lengths else 0
    min_length = min(lengths) if lengths else 0
    max_length = max(lengths) if lengths else 0
    
    print(f"Average sequence length: {avg_length:.1f}")
    print(f"Minimum sequence length: {min_length}")
    print(f"Maximum sequence length: {max_length}")
    
    # Process sequences in batches to avoid memory issues
    batch_size = 2  # In real applications, this would be much larger
    for i in range(0, sequence_count, batch_size):
        batch = msa_dataset.sequences[i:i+batch_size]
        # Process batch...
        print(f"Processed batch {i//batch_size + 1} with {len(batch)} sequences")
except Exception as e:
    print(f"Error working with large MSA: {e}")

## Best Practices

1. **File Format Selection**: Choose the appropriate MSA format based on your needs
   - A2M: When you need guaranteed equal sequence lengths
   - A3M: When working with insertions relative to a query
   - PSI: For simpler, space-separated formats
2. **Memory Management**: Be aware that large MSAs can consume significant memory
3. **Path Configuration**: Use absolute paths or ensure relative paths are correct
4. **Error Handling**: Add try-except blocks when loading MSA files to handle potential format issues

## Troubleshooting

Common issues when working with MSA datasets:

1. **File Not Found**: Ensure the path to your MSA file is correct
2. **Format Errors**: Verify that your file follows the expected format conventions
3. **Inconsistent Sequence Lengths**: For A2M files, all sequences must have the same length
4. **Memory Issues**: Large MSAs may cause memory problems; consider processing data in batches

## Future Developments

The current implementation of `MSADataset` has some limitations:

1. Directory support is not yet implemented
2. Split functionality (train/valid/test) is not yet available
3. Additional MSA formats could be supported in the future

## Summary

The MSA dataset provides a straightforward way to work with multiple sequence alignment data in the PG2 dataset system. By supporting multiple file formats (A2M, A3M, PSI), it allows you to work with alignments from various sources. The simple interface makes it easy to access and analyze the aligned sequences for evolutionary analysis, structure prediction, or other bioinformatics applications.

In [ ]:
# Clean up the files we created
import os

try:
    os.remove("sample_msa_dataset.toml")
    os.remove("../example_data/sample_alignment.a3m")
    os.remove("../example_data/sample_alignment.a2m")
    os.remove("../example_data/sample_alignment.psi")
    os.remove("../example_data/sample_data.csv")
    print("Cleaned up sample files")
except Exception as e:
    print(f"Error cleaning up: {e}")